# 4-Qubit Max-Cut QAOA: Implementation Paper Notebook
**Companion implementation to:** *Quantum Approximate Optimization Algorithms: A Mathematical Survey*  
Mithilesh Kumar & Sanjana Mattaparthi, Krea University, Sri City

This notebook provides a complete, reproducible implementation of the QAOA pipeline for the 4-qubit Max-Cut problem on the cycle graph $C_4$, **plus** a generalised QAOA framework covering:

1. **Environment & Versions** — package pinning and reproducibility setup
2. **Problem Setup** — graph definition and classical brute-force reference
3. **Hamiltonian Construction** — encoding Max-Cut as $H_C$
4. **Ideal QAOA Circuit** — $p$-layer ansatz and exact statevector evaluation
5. **Parameter-Shift Gradient** — exact gradient via Eq. 3.23 of the survey
6. **COBYLA Multi-restart Optimisation** — Survey Algorithm 1
7. **Monotonicity** — Survey Theorem 3.3
8. **INTERP Warm-Start** — depth $p{-}1 \to p$ initialisation (Zhou et al. 2020)
9. **Probability Distribution Analysis** — statevector measurement distribution
10. **Hardware Transpilation** — baseline vs. optimised compilation to FakeManilaV2
11. **Noise Model Validation** — $F^\text{noisy}_p \approx (1-\varepsilon)^{G_p}F^\text{ideal}_p + [1-(1-\varepsilon)^{G_p}]\bar{C}$
12. **Ideal vs. Noisy Simulation** — shot-based comparison at 4096 shots
13. **Connection to Theory** — Theorem IV.3 lower bound and GW benchmark
14. **Summary** — all results in one table and figure
15. **Generalised QAOA Framework** — problem-agnostic pipeline for arbitrary QUBO/Ising problems

```
pip install qiskit==1.0.2 numpy==1.26.4 scipy==1.14.1 qiskit-aer==0.14.2 qiskit-ibm-runtime matplotlib
```

In [ ]:
!pip install qiskit==1.0.2 numpy==1.26.4 scipy==1.14.1 qiskit-aer==0.14.2 qiskit-ibm-runtime matplotlib


## 1. Environment & Version Check

Pinning package versions ensures reproducibility across runs and machines.

In [ ]:
from __future__ import annotations

import itertools
import math
import time
import warnings
from dataclasses import dataclass, field
from statistics import mean
from typing import Any, Callable, Optional, Sequence

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.linalg import expm
from scipy.optimize import minimize

import qiskit
import qiskit_aer
from qiskit import QuantumCircuit, transpile
from qiskit.qasm2 import dumps
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

warnings.filterwarnings('ignore')

# ── Global seeds for full reproducibility ─────────────────────────────
SEED            = 42
SEED_SIMULATOR  = 42
SEED_TRANSPILER = 123
np.random.seed(SEED)

print('Package versions (pinned):')
print(f'  qiskit     : {qiskit.__version__}')
print(f'  qiskit-aer : {qiskit_aer.__version__}')
print(f'  numpy      : {np.__version__}')
print('All imports OK — seeds fixed for reproducibility.')


## 2. Problem Definition

We work with the 4-node cycle graph $C_4$:
$$G=(V,E),\quad V=\{0,1,2,3\},\quad E=\{(0,1),(1,2),(2,3),(3,0)\}$$
The Max-Cut objective for a bitstring $z=z_0z_1z_2z_3$ is:
$$C(z)=\sum_{(i,j)\in E}\mathbf{1}[z_i\ne z_j]$$
The classical maximum is $C^*=4$, achieved by bitstrings `0101` and `1010`.

In [ ]:
NUM_QUBITS          = 4
EDGES: list[tuple[int,int]] = [(0,1),(1,2),(2,3),(3,0)]
MAX_CLASSICAL_CUT   = 4
C_BAR               = len(EDGES) / 2.0   # uniform partition baseline = 2.0
GW_BOUND            = 0.8786             # Goemans-Williamson ratio
THEORY_BOUND_P1     = 0.6924             # Survey Theorem IV.3

def maxcut_cost(bitstring: str, edges=EDGES) -> int:
    bits = bitstring[::-1]
    return sum(1 for i,j in edges if bits[i] != bits[j])

def brute_force_solutions(num_qubits=NUM_QUBITS, edges=EDGES):
    best_val, best_bs = -1, []
    for bits in itertools.product('01', repeat=num_qubits):
        bs = ''.join(bits)
        cv = maxcut_cost(bs, edges)
        if cv > best_val:  best_val=cv; best_bs=[bs]
        elif cv == best_val: best_bs.append(bs)
    return best_val, best_bs

classical_cut, classical_solutions = brute_force_solutions()
print(f'Graph    : C_4 (4-node cycle)')
print(f'C*       : {classical_cut}')
print(f'Optimal  : {classical_solutions}')
print(f'C_bar    : {C_BAR}')


## 3. Hamiltonian Construction

The Max-Cut cost Hamiltonian (Survey Eq. 26):
$$H_C=\frac{1}{2}\sum_{(i,j)\in E}(I-Z_iZ_j)$$

In [ ]:
def maxcut_hamiltonian(num_qubits=NUM_QUBITS, edges=EDGES):
    pauli_terms = [('I'*num_qubits, 0.5*len(edges))]
    for i,j in edges:
        label = ['I']*num_qubits
        label[num_qubits-1-i] = 'Z'
        label[num_qubits-1-j] = 'Z'
        pauli_terms.append((''.join(label), -0.5))
    return SparsePauliOp.from_list(pauli_terms)

H_C = maxcut_hamiltonian()
eigenvalues = sorted(set(np.real(np.linalg.eigvalsh(H_C.to_matrix()))))
print('H_C:', H_C)
print(f'Eigenvalues : {eigenvalues}  (= possible cut values)')
print(f'Max eigenvalue : {max(eigenvalues):.1f} = C* = {classical_cut}')


## 4. QAOA Circuit Construction

The depth-$p$ QAOA ansatz (Survey Eq. 14):
$$|\psi_p(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle = \prod_{k=1}^p e^{-i\beta_k H_B}e^{-i\gamma_k H_C}|+\rangle^{\otimes 4}$$
- **Cost layer** — each edge $(i,j)$: `CX–RZ(–γ)–CX` implements $e^{i(\gamma/2)Z_iZ_j}$
- **Mixer layer** — each qubit $k$: `RX(2β)` implements $e^{-i\beta X_k}$

In [ ]:
def build_qaoa_circuit(params, num_qubits=NUM_QUBITS, edges=EDGES, measure=False):
    p      = len(params) // 2
    gammas = params[:p]
    betas  = params[p:]
    circuit = QuantumCircuit(num_qubits, num_qubits if measure else 0)
    circuit.h(range(num_qubits))
    for layer in range(p):
        circuit.barrier(label=f'cost_{layer+1}')
        for ctrl, tgt in edges:
            circuit.cx(ctrl, tgt)
            circuit.rz(-gammas[layer], tgt)
            circuit.cx(ctrl, tgt)
        circuit.barrier(label=f'mix_{layer+1}')
        for q in range(num_qubits):
            circuit.rx(2.0*betas[layer], q)
    if measure:
        circuit.barrier()
        circuit.measure(range(num_qubits), range(num_qubits))
    return circuit

def circuit_stats(circuit):
    ops   = circuit.count_ops()
    two_q = sum(int(ops.get(g,0)) for g in ('cx','cz','ecr','swap','rzz'))
    return {'depth':circuit.depth(),'size':circuit.size(),'two_qubit_gates':two_q,'ops':dict(ops)}

def expectation_ideal(params, H=H_C):
    sv = Statevector.from_instruction(build_qaoa_circuit(params))
    return float(np.real(sv.expectation_value(H)))

# ── Circuit depth table ───────────────────────────────────────────────
print(f"{'p':>3} {'depth':>7} {'size':>6} {'2Q gates':>10}")
print('─'*30)
for p in [1,2,3]:
    s = circuit_stats(build_qaoa_circuit(np.zeros(2*p)))
    print(f"{p:>3} {s['depth']:>7} {s['size']:>6} {s['two_qubit_gates']:>10}")
print()
print('QAOA p=1 circuit (OpenQASM 2.0):')
print('─'*60)
print(dumps(build_qaoa_circuit(np.array([math.pi*3/4, math.pi*3/8]), measure=True)))


## 5. Parameter-Shift Gradient

Exact gradient via the parameter-shift rule (Survey Eq. 3.23). For generators with eigenvalues $\pm r=\pm\tfrac{1}{2}$:
$$\frac{\partial F_p}{\partial\theta_k}=\frac{1}{2}\left[F_p\!\left(\theta_k+\frac{\pi}{4}\right)-F_p\!\left(\theta_k-\frac{\pi}{4}\right)\right]$$
Cost: exactly $4p$ circuit evaluations — no finite-difference approximation.

In [ ]:
def parameter_shift_gradient(params, H=H_C):
    grad = np.zeros_like(params)
    for i in range(len(params)):
        pp = params.copy(); pp[i] += math.pi/4
        pm = params.copy(); pm[i] -= math.pi/4
        grad[i] = (expectation_ideal(pp,H) - expectation_ideal(pm,H)) / 2
    return grad

optimal_p1   = np.array([math.pi*3/4, math.pi*3/8])
grad_at_opt  = parameter_shift_gradient(optimal_p1)
print(f'Gradient at analytic optimum (3π/4, 3π/8):')
print(f'  ∂F/∂γ = {grad_at_opt[0]:.2e}  (≈ 0 ✓)')
print(f'  ∂F/∂β = {grad_at_opt[1]:.2e}  (≈ 0 ✓)')
print()
grad_off = parameter_shift_gradient(np.array([0.5, 1.0]))
print(f'Gradient at off-optimum (0.5, 1.0):')
print(f'  ∂F/∂γ = {grad_off[0]:+.6f}')
print(f'  ∂F/∂β = {grad_off[1]:+.6f}')


## 6. COBYLA Multi-restart Optimisation

We maximise $F_p(\boldsymbol{\gamma},\boldsymbol{\beta})$ using COBYLA with 5 random restarts (Survey Algorithm 1).

In [ ]:
def optimise_qaoa(p, n_restarts=5, H=H_C, rng=None):
    if rng is None: rng = np.random.default_rng(SEED)
    def objective(params): return -expectation_ideal(params, H)
    best_params, best_neg, history = None, float('inf'), []
    for _ in range(n_restarts):
        x0  = rng.uniform(0, math.pi, 2*p)
        res = minimize(objective, x0, method='COBYLA',
                       options={'maxiter':500,'rhobeg':0.5})
        history.append(-res.fun)
        if res.fun < best_neg: best_neg=res.fun; best_params=res.x
    return best_params, -best_neg, history

rng = np.random.default_rng(SEED)
opt_results = {}
print(f"{'p':>3} {'F_p':>10} {'α':>10} {'time':>8} {'restarts':>10}")
print('─'*50)
for p in [1,2,3]:
    t0 = time.perf_counter()
    bp, bF, hist = optimise_qaoa(p, n_restarts=5, rng=rng)
    elapsed = time.perf_counter()-t0
    alpha = bF/MAX_CLASSICAL_CUT
    opt_results[p] = {'params':bp,'Fp':bF,'alpha':alpha,'time':elapsed,'history':hist}
    print(f"{p:>3} {bF:>10.6f} {alpha:>10.6f} {elapsed:>7.2f}s {len(hist):>10}")


## 7. Monotonicity $M_p\ge M_{p-1}$ — Survey Theorem 3.3

In [ ]:
depths = sorted(opt_results.keys())
alphas = [opt_results[p]['alpha'] for p in depths]
mono   = all(alphas[i] <= alphas[i+1]+1e-6 for i in range(len(alphas)-1))
print(f'Monotonicity M_p >= M_{{p-1}}: {mono} ({"✓ Theorem 3.3 confirmed" if mono else "✗"})')
for p,a in zip(depths,alphas):
    print(f'  p={p}: α={a:.6f} {"█"*int(a*40)}')

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(depths, alphas, 'o-', color='#4C72B0', lw=2.5, ms=9, label='QAOA (this work)')
ax.axhline(THEORY_BOUND_P1, ls='--', color='#888', lw=1.5, label=f'Theory bound ({THEORY_BOUND_P1})')
ax.axhline(GW_BOUND, ls='--', color='#2ca02c', lw=1.5, label=f'GW ({GW_BOUND})')
ax.axhline(1.0, ls=':', color='red', lw=1.5, label='Optimal (α=1)')
for p,a in zip(depths,alphas):
    ax.annotate(f'α={a:.4f}', (p,a), textcoords='offset points', xytext=(8,4), fontsize=9)
ax.set_xlabel('Depth $p$', fontsize=12)
ax.set_ylabel('Approximation ratio $\\alpha_p$', fontsize=12)
ax.set_title('Ideal QAOA — Monotonicity $M_p \\geq M_{p-1}$ (Theorem 3.3)', fontsize=11)
ax.set_xticks(depths); ax.set_ylim(0.65, 1.08)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 8. INTERP Warm-Start — Zhou et al. (2020)

INTERP linearly interpolates optimal depth-$(p-1)$ parameters to initialise depth $p$, reducing optimiser iterations by 3–5× (Survey Proposition 3.6).

In [ ]:
def interp_init(gamma_prev, beta_prev):
    p_prev, p_new = len(gamma_prev), len(gamma_prev)+1
    gamma_new, beta_new = np.zeros(p_new), np.zeros(p_new)
    for k in range(1, p_new+1):
        frac   = (k-1)/(p_new-1) if p_new>1 else 0.0
        idx_lo = min(int(frac*(p_prev-1)), p_prev-2) if p_prev>1 else 0
        idx_hi = idx_lo+1 if p_prev>1 else 0
        w      = frac*(p_prev-1)-idx_lo if p_prev>1 else 0.0
        gamma_new[k-1] = (1-w)*gamma_prev[idx_lo]+w*gamma_prev[min(idx_hi,p_prev-1)]
        beta_new[k-1]  = (1-w)*beta_prev[idx_lo] +w*beta_prev[min(idx_hi,p_prev-1)]
    return gamma_new, beta_new

rng2 = np.random.default_rng(SEED)
t0 = time.perf_counter()
_, Fp_random, _ = optimise_qaoa(p=2, n_restarts=5, rng=rng2)
t_random = time.perf_counter()-t0

p1_params = opt_results[1]['params']
g0, b0    = interp_init(p1_params[:1], p1_params[1:])
init_interp = np.concatenate([g0, b0])

t0 = time.perf_counter()
res_interp = minimize(lambda p: -expectation_ideal(p), init_interp,
                      method='COBYLA', options={'maxiter':500,'rhobeg':0.5})
Fp_interp = -res_interp.fun
t_interp  = time.perf_counter()-t0

print(f'INTERP warm-start comparison (p=1 → p=2):')
print(f'  p=1 optimum: γ*={p1_params[0]:.4f}, β*={p1_params[1]:.4f}')
print(f'  INTERP init: γ={np.round(init_interp[:2],4)}, β={np.round(init_interp[2:],4)}')
print()
print(f'  Random init (5 restarts): F_2={Fp_random:.6f}, t={t_random:.2f}s')
print(f'  INTERP (1 run):           F_2={Fp_interp:.6f}, t={t_interp:.2f}s')
print(f'  Speedup: {t_random/t_interp:.2f}×  |  Quality diff: {Fp_interp-Fp_random:+.6f}')


## 9. Statevector Distribution at Optimal Parameters

In [ ]:
all_states = [f'{i:04b}' for i in range(16)]
cuts       = [maxcut_cost(s) for s in all_states]
optimal    = [c == MAX_CLASSICAL_CUT for c in cuts]

fig, axes = plt.subplots(1,2, figsize=(14,4))
for ax, p in zip(axes, [1,2]):
    sv    = Statevector.from_instruction(build_qaoa_circuit(opt_results[p]['params']))
    pd    = sv.probabilities_dict()
    probs = [pd.get(s,0.0) for s in all_states]
    bc    = ['#2ca02c' if o else '#1f77b4' for o in optimal]
    ax.bar(all_states, probs, color=bc, edgecolor='white', linewidth=0.5)
    p_opt = sum(pr for pr,o in zip(probs,optimal) if o)
    ax.set_title(f'$p={p}$: $\\langle H_C\\rangle={opt_results[p]["Fp"]:.4f}$, '
                 f'$\\alpha={opt_results[p]["alpha"]:.4f}$, $P(\\text{{opt}})={p_opt:.3f}$',
                 fontsize=10)
    ax.set_xlabel('Basis state', fontsize=9); ax.set_ylabel('Probability', fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
axes[0].legend(handles=[Patch(facecolor='#2ca02c',label='Optimal (C=4)'),
                         Patch(facecolor='#1f77b4',label='Sub-optimal')], fontsize=9)
plt.suptitle('Ideal QAOA Probability Distributions — $p=1$ vs $p=2$', fontsize=12)
plt.tight_layout(); plt.show()

print('\nStatevector table at p=1:')
sv1   = Statevector.from_instruction(build_qaoa_circuit(opt_results[1]['params']))
pd1   = sv1.probabilities_dict()
probs1 = [pd1.get(s,0.0) for s in all_states]
print(f'{"Bitstring":>12} {"Cut":>5} {"Probability":>12} {"Optimal?":>10}')
print('─'*44)
for s,pr,c,o in sorted(zip(all_states,probs1,cuts,optimal), key=lambda x:-x[2]):
    print(f'{s:>12} {c:>5} {pr:>10.4f} {"✓" if o else "":>8}')
print(f'\nP(opt) at p=1: {sum(p for p,o in zip(probs1,optimal) if o):.4f}')


## 10. Hardware Transpilation

In [ ]:
backend   = FakeManilaV2()
p1_circuit = build_qaoa_circuit(opt_results[1]['params'], measure=True)
t0       = time.perf_counter()
baseline = transpile(p1_circuit, backend=backend, optimization_level=0)
t_base   = time.perf_counter()-t0
t0       = time.perf_counter()
pm       = generate_preset_pass_manager(backend=backend, optimization_level=3,
                                        layout_method='sabre', routing_method='sabre',
                                        seed_transpiler=SEED_TRANSPILER)
optimised = pm.run(p1_circuit)
t_opt    = time.perf_counter()-t0

logical_m = circuit_stats(p1_circuit)
base_m    = circuit_stats(baseline)
opt_m     = circuit_stats(optimised)

print(f'{"Metric":<22} {"Logical":>10} {"Baseline":>12} {"Optimised":>12}')
print('─'*60)
for key,lbl in [('depth','Depth'),('size','Total ops'),('two_qubit_gates','2Q gates')]:
    print(f'{lbl:<22} {logical_m[key]:>10} {base_m[key]:>12} {opt_m[key]:>12}')
reduction = (1-opt_m['two_qubit_gates']/base_m['two_qubit_gates'])*100
print(f'\n2Q gate reduction: {base_m["two_qubit_gates"]} → {opt_m["two_qubit_gates"]} ({reduction:.0f}% fewer)')


## 11. Noise Model Validation

Survey Eq. 7.1: $F^\text{noisy}_p\approx(1-\varepsilon)^{G_p}F^\text{ideal}_p+[1-(1-\varepsilon)^{G_p}]\bar{C}$

In [ ]:
EPS = 2e-3
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(1e-4,1),['rz','rx','h','x'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(EPS,2),['cx'])
noisy_sim = AerSimulator(noise_model=noise_model)
SHOTS = 4096

Fp_ideal_list, Fp_noisy_list, Fp_pred_list, G_p_list = [], [], [], []
print(f'Noise model: ε={EPS}, C̄={C_BAR}')
print(f'{"p":>3} {"G_p":>6} {"F_ideal":>10} {"F_noisy":>10} {"F_pred":>10} {"MAE":>8}')
print('─'*55)
for p in [1,2,3]:
    params = opt_results[p]['params']; Fp_i = opt_results[p]['Fp']
    qc_t   = transpile(build_qaoa_circuit(params,measure=True),
                       basis_gates=['cx','rz','rx','h','x'],
                       optimization_level=3, seed_transpiler=SEED_TRANSPILER)
    G_p    = qc_t.count_ops().get('cx',0); G_p_list.append(G_p)
    counts = noisy_sim.run(qc_t,shots=SHOTS,seed_simulator=SEED_SIMULATOR).result().get_counts()
    total  = sum(counts.values())
    Fp_n   = sum(cnt/total*maxcut_cost(bs) for bs,cnt in counts.items())
    Fp_pr  = (1-EPS)**G_p*Fp_i + (1-(1-EPS)**G_p)*C_BAR
    Fp_ideal_list.append(Fp_i); Fp_noisy_list.append(Fp_n); Fp_pred_list.append(Fp_pr)
    print(f'{p:>3} {G_p:>6} {Fp_i:>10.4f} {Fp_n:>10.4f} {Fp_pr:>10.4f} {abs(Fp_n-Fp_pr):>8.4f}')
MAE    = np.mean(np.abs(np.array(Fp_noisy_list)-np.array(Fp_pred_list)))
p_star = [1,2,3][np.argmax(Fp_noisy_list)]
print(f'\nMAE = {MAE:.4f}  |  Empirical p* = {p_star}')


## 12. Ideal vs. Noisy Simulation — Shot-Based Comparison

In [ ]:
ideal_sim = AerSimulator()
p1_meas   = build_qaoa_circuit(opt_results[1]['params'], measure=True)
ideal_counts = ideal_sim.run(transpile(p1_meas,ideal_sim),
                              shots=SHOTS,seed_simulator=SEED_SIMULATOR).result().get_counts()
noisy_counts = noisy_sim.run(transpile(p1_meas,basis_gates=['cx','rz','rx','h','x'],
                                        optimization_level=3,seed_transpiler=SEED_TRANSPILER),
                              shots=SHOTS,seed_simulator=SEED_SIMULATOR).result().get_counts()

def summarize_counts(counts):
    cuts_exp = []
    best_cut, best_bs = -1, []
    for bs,cnt in counts.items():
        c = maxcut_cost(bs); cuts_exp.extend([c]*cnt)
        if c>best_cut: best_cut=c; best_bs=[bs]
        elif c==best_cut: best_bs.append(bs)
    mf = max(counts, key=counts.get)
    return dict(most_frequent=mf, most_frequent_count=counts[mf],
                best_cut=best_cut, avg_cut=mean(cuts_exp), unique=len(counts))

ideal_s = summarize_counts(ideal_counts)
noisy_s = summarize_counts(noisy_counts)
print(f'{"":38} {"Ideal":>12} {"Noisy":>12}')
print('─'*65)
for lbl,iv,nv in [
    ('Most frequent bitstring',ideal_s['most_frequent'],noisy_s['most_frequent']),
    ('Most frequent count',ideal_s['most_frequent_count'],noisy_s['most_frequent_count']),
    ('Best sampled cut',ideal_s['best_cut'],noisy_s['best_cut']),
    ('Average cut ⟨C⟩',f"{ideal_s['avg_cut']:.4f}",f"{noisy_s['avg_cut']:.4f}"),
    ('Approx. ratio α',f"{ideal_s['avg_cut']/MAX_CLASSICAL_CUT:.4f}",
     f"{noisy_s['avg_cut']/MAX_CLASSICAL_CUT:.4f}"),
    ('Unique bitstrings',ideal_s['unique'],noisy_s['unique']),
]:
    print(f'{lbl:<38} {str(iv):>12} {str(nv):>12}')
noise_deg = ideal_s['avg_cut']-noisy_s['avg_cut']
print(f'\nNoise degradation: {noise_deg:.4f} ({100*noise_deg/ideal_s["avg_cut"]:.1f}%)')


## 13. Connection to Theory — Theorem IV.3 and the GW Bound

In [ ]:
alpha_ideal_p1   = opt_results[1]['alpha']
alpha_ideal_p2   = opt_results[2]['alpha']
alpha_noisy      = noisy_s['avg_cut'] / MAX_CLASSICAL_CUT
alpha_ideal_shot = ideal_s['avg_cut'] / MAX_CLASSICAL_CUT

print('Approximation ratio hierarchy:')
print(f'  Random baseline    : 0.5000')
print(f'  Theory bound (p=1) : {THEORY_BOUND_P1}  [Survey Theorem IV.3]')
print(f'  QAOA p=1 noisy     : {alpha_noisy:.4f}')
print(f'  QAOA p=1 ideal sim : {alpha_ideal_shot:.4f}')
print(f'  QAOA p=1 exact     : {alpha_ideal_p1:.4f}')
print(f'  Goemans-Williamson : {GW_BOUND}')
print(f'  QAOA p=2 exact     : {alpha_ideal_p2:.4f}  ← optimal!')
print(f'\nMargin above Theorem IV.3: {alpha_noisy-THEORY_BOUND_P1:+.4f}')


## 14. Complete Results Summary

In [ ]:
print('='*65)
print('  4-QUBIT MAX-CUT QAOA — COMPLETE RESULTS SUMMARY')
print('='*65)
for p in [1,2,3]:
    r = opt_results[p]
    print(f'  p={p}: F={r["Fp"]:.6f}, α={r["alpha"]:.6f}, t={r["time"]:.2f}s')
print(f'  Monotonicity confirmed: {[opt_results[p]["Fp"] for p in [1,2,3]]}')
print(f'  |∂F/∂γ| at optimum : {abs(grad_at_opt[0]):.2e}')
print(f'  INTERP speedup      : {t_random/t_interp:.2f}×')
print(f'  Noise MAE           : {MAE:.4f}')
print(f'  Empirical p*        : {p_star}')
print('='*65)


---
## 15. Generalised QAOA Framework

This section lifts the entire $C_4$ Max-Cut pipeline into a **problem-agnostic generalised QAOA algorithm**. The only problem-specific inputs are:
- `H_C` — a `SparsePauliOp` encoding the cost Hamiltonian $H_C$ with $H_C|z\rangle = C(z)|z\rangle$
- `cost_fn` — a classical Python function $C:\{0,1\}^n\to\mathbb{R}$

Everything else (circuit construction, gradient, optimisation, INTERP, noise model, shot simulation) is fully general.

**Supported problems:** Max-Cut (weighted/unweighted), Number Partitioning, Weighted Max-SAT, any QUBO/Ising problem.

### 15.1 Data Structures: `QAOAConfig` and `QAOAResult`

In [ ]:
@dataclass
class QAOAConfig:
    """All inputs to the generalised QAOA pipeline."""
    H_C:                SparsePauliOp
    cost_fn:            Callable[[str], float]
    n_qubits:           int
    p:                  int                         = 1
    mixer:              Any                         = 'X'
    optimizer:          str                         = 'COBYLA'
    n_restarts:         int                         = 5
    warm_start_params:  Optional[np.ndarray]        = None
    C_opt:              Optional[float]             = None
    backend:            Any                         = None
    noise_eps:          float                       = 0.0
    shots:              int                         = 4096
    seed:               int                         = 42
    seed_transpiler:    int                         = 123
    param_bounds:       tuple                       = (0.0, math.pi)

@dataclass
class QAOAResult:
    """All outputs of the generalised QAOA pipeline."""
    optimal_params:             np.ndarray
    optimal_F:                  float
    alpha:                      float
    gradient_at_opt:            np.ndarray
    statevector:                Statevector
    prob_distribution:          dict
    top_k_bitstrings:           list
    shot_counts_ideal:          dict
    shot_avg_cost_ideal:        float
    shot_counts_noisy:          Optional[dict]   = None
    shot_avg_cost_noisy:        Optional[float]  = None
    noise_prediction:           Optional[float]  = None
    transpiled_circuit:         Any              = None
    transpilation_cx_reduction: Optional[float]  = None
    interp_params_next:         np.ndarray       = field(default_factory=lambda: np.array([]))
    elapsed_time:               float            = 0.0
    n_circuit_evaluations:      int              = 0
    brute_force_optimum:        Optional[float]  = None
    brute_force_solutions:      Optional[list]   = None

print('QAOAConfig and QAOAResult dataclasses defined.')


### 15.2 Brute-Force Reference and Hamiltonian Utilities

In [ ]:
def brute_force_general(cost_fn, n_qubits):
    """O(2^n) classical reference."""
    best_val, best_strs = -math.inf, []
    for bits in itertools.product('01', repeat=n_qubits):
        bs = ''.join(bits); v = cost_fn(bs)
        if v > best_val:   best_val, best_strs = v, [bs]
        elif v == best_val: best_strs.append(bs)
    return best_val, best_strs

def hamiltonian_eigenvalues_gen(H_C):
    eigs = np.linalg.eigvalsh(H_C.to_matrix())
    return np.unique(np.round(np.real(eigs), 10))

print('Brute-force and eigenvalue utilities defined.')


### 15.3 Generalised Mixer and Circuit Construction

In [ ]:
def build_mixer_general(mixer, n_qubits):
    """Build H_B: 'X' (standard), 'XY' (constrained), or custom SparsePauliOp."""
    if isinstance(mixer, SparsePauliOp): return mixer
    n = n_qubits
    if mixer == 'X':
        terms = []
        for j in range(n):
            label = ['I']*n; label[n-1-j]='X'
            terms.append((''.join(label), 1.0))
        return SparsePauliOp.from_list(terms)
    if mixer == 'XY':
        terms = []
        for j in range(n-1):
            lx=['I']*n; lx[n-1-j]='X'; lx[n-2-j]='X'; terms.append((''.join(lx),1.0))
            ly=['I']*n; ly[n-1-j]='Y'; ly[n-2-j]='Y'; terms.append((''.join(ly),1.0))
        return SparsePauliOp.from_list(terms)
    raise ValueError(f"Unknown mixer '{mixer}'. Use 'X', 'XY', or SparsePauliOp.")

def apply_pauli_rotation(qc, pauli_str, coeff, angle, n_qubits):
    """Apply e^{-i*angle*coeff*P} for a single Pauli string P via CX-ladder."""
    ops = list(reversed(pauli_str))   # ops[q] = Pauli on qubit q
    z_q = [q for q,op in enumerate(ops) if op=='Z']
    x_q = [q for q,op in enumerate(ops) if op=='X']
    y_q = [q for q,op in enumerate(ops) if op=='Y']
    active = z_q + x_q + y_q
    if not active: return
    for q in x_q: qc.h(q)
    for q in y_q: qc.sdg(q); qc.h(q)
    target = active[-1]
    for ctrl in active[:-1]: qc.cx(ctrl, target)
    qc.rz(-2.0 * coeff * angle, target)
    for ctrl in reversed(active[:-1]): qc.cx(ctrl, target)
    for q in x_q: qc.h(q)
    for q in y_q: qc.h(q); qc.s(q)

def build_qaoa_circuit_general(params, H_C, H_B, n_qubits, measure=False):
    """General QAOA circuit: works for any SparsePauliOp cost + mixer."""
    p       = len(params) // 2
    gammas  = params[:p]
    betas   = params[p:]
    qc      = QuantumCircuit(n_qubits, n_qubits if measure else 0)
    qc.h(range(n_qubits))
    for layer in range(p):
        qc.barrier(label=f'cost_{layer+1}')
        for pauli_str, coeff in zip(H_C.paulis.to_labels(), H_C.coeffs):
            apply_pauli_rotation(qc, pauli_str, float(np.real(coeff)), gammas[layer], n_qubits)
        qc.barrier(label=f'mix_{layer+1}')
        for pauli_str, coeff in zip(H_B.paulis.to_labels(), H_B.coeffs):
            apply_pauli_rotation(qc, pauli_str, float(np.real(coeff)), betas[layer], n_qubits)
    if measure:
        qc.barrier(); qc.measure(range(n_qubits), range(n_qubits))
    return qc

def expectation_general(params, H_C, H_B, n_qubits):
    sv = Statevector.from_instruction(build_qaoa_circuit_general(params, H_C, H_B, n_qubits))
    return float(np.real(sv.expectation_value(H_C)))

print('Generalised circuit and mixer construction defined.')


### 15.4 Generalised Gradient, Optimiser, and INTERP

In [ ]:
def param_shift_grad_general(params, H_C, H_B, n_qubits, r=0.5):
    """Exact parameter-shift gradient. Cost: 4p circuit evaluations."""
    shift = math.pi / (4.0*r)
    grad  = np.zeros_like(params, dtype=float)
    for i in range(len(params)):
        pp = params.copy(); pp[i] += shift
        pm = params.copy(); pm[i] -= shift
        grad[i] = r*(expectation_general(pp,H_C,H_B,n_qubits)
                    -expectation_general(pm,H_C,H_B,n_qubits))
    return grad

def optimise_general(H_C, H_B, n_qubits, p, optimizer, n_restarts,
                     warm_start, param_bounds, seed):
    """Multi-restart optimiser (COBYLA / L-BFGS-B / ADAM)."""
    rng     = np.random.default_rng(seed)
    n_evals = [0]
    def obj(params): n_evals[0]+=1; return -expectation_general(params,H_C,H_B,n_qubits)
    def grad(params): return -param_shift_grad_general(params,H_C,H_B,n_qubits)
    inits   = [warm_start] if warm_start is not None else\
              [rng.uniform(*param_bounds, 2*p) for _ in range(n_restarts)]
    best_params, best_neg, history = None, math.inf, []
    for x0 in inits:
        if optimizer == 'COBYLA':
            res = minimize(obj, x0, method='COBYLA', options={'maxiter':1000,'rhobeg':0.5})
        elif optimizer == 'L-BFGS-B':
            res = minimize(obj, x0, jac=grad, method='L-BFGS-B',
                           bounds=[param_bounds]*(2*p), options={'maxiter':500,'ftol':1e-12})
        else:
            raise ValueError(f'Unknown optimizer: {optimizer}')
        history.append(-res.fun)
        if res.fun < best_neg: best_neg=res.fun; best_params=res.x
    return best_params, -best_neg, history, n_evals[0]

def interp_init_general(optimal_params):
    """INTERP warm-start: depth p → p+1 (Zhou et al. 2020 / Survey Prop. 3.6)."""
    p       = len(optimal_params)//2
    gammas  = optimal_params[:p]
    betas   = optimal_params[p:]
    p_new   = p+1
    g_new   = np.zeros(p_new); b_new = np.zeros(p_new)
    for k in range(1, p_new+1):
        frac   = (k-1)/(p_new-1) if p_new>1 else 0.0
        idx_lo = min(int(frac*(p-1)), max(p-2,0))
        idx_hi = min(idx_lo+1, p-1) if p>1 else 0
        w      = frac*(p-1)-idx_lo if p>1 else 0.0
        g_new[k-1] = (1-w)*gammas[idx_lo]+w*gammas[idx_hi]
        b_new[k-1] = (1-w)*betas[idx_lo] +w*betas[idx_hi]
    return np.concatenate([g_new, b_new])

print('Generalised gradient, optimiser, and INTERP defined.')


### 15.5 Main Pipeline: `run_qaoa_general(config)`

In [ ]:
def run_qaoa_general(cfg: QAOAConfig) -> QAOAResult:
    """Execute the complete generalised QAOA pipeline."""
    np.random.seed(cfg.seed)
    t0 = time.perf_counter()
    n  = cfg.n_qubits

    # Step 1: Brute-force reference
    bf_opt, bf_sols = None, None
    if n <= 20: bf_opt, bf_sols = brute_force_general(cfg.cost_fn, n)
    C_opt = cfg.C_opt if cfg.C_opt is not None else (bf_opt if bf_opt is not None
            else float(np.max(np.real(np.linalg.eigvalsh(cfg.H_C.to_matrix())))))

    # Step 2: Mixer
    H_B = build_mixer_general(cfg.mixer, n)

    # Step 3-4: Optimisation
    best_params, best_F, history, n_evals = optimise_general(
        cfg.H_C, H_B, n, cfg.p, cfg.optimizer, cfg.n_restarts,
        cfg.warm_start_params, cfg.param_bounds, cfg.seed)

    # Step 5: Gradient check
    grad = param_shift_grad_general(best_params, cfg.H_C, H_B, n)
    n_evals += 2*len(best_params)

    # Step 6: Approximation ratio
    alpha = best_F / C_opt if C_opt != 0 else float('nan')

    # Step 7: Statevector analysis
    circuit  = build_qaoa_circuit_general(best_params, cfg.H_C, H_B, n)
    sv       = Statevector.from_instruction(circuit)
    pd       = sv.probabilities_dict()
    all_bs   = [f'{i:0{n}b}' for i in range(2**n)]
    prob_dist = {bs: float(pd.get(bs,0.0)) for bs in all_bs}
    ranked   = sorted([(bs,cfg.cost_fn(bs),prob_dist[bs]) for bs in all_bs],
                      key=lambda x:x[2], reverse=True)

    # Step 8: Shot simulation
    circuit_m   = build_qaoa_circuit_general(best_params, cfg.H_C, H_B, n, measure=True)
    ideal_sim   = AerSimulator()
    ideal_counts = ideal_sim.run(transpile(circuit_m, ideal_sim),
                                  shots=cfg.shots, seed_simulator=cfg.seed).result().get_counts()
    ideal_avg   = sum((cnt/cfg.shots)*cfg.cost_fn(bs) for bs,cnt in ideal_counts.items())
    noisy_counts, noisy_avg, noise_pred = None, None, None
    if cfg.noise_eps > 0:
        nm = NoiseModel()
        nm.add_all_qubit_quantum_error(depolarizing_error(1e-4,1),['rz','rx','h','x','sx'])
        nm.add_all_qubit_quantum_error(depolarizing_error(cfg.noise_eps,2),['cx'])
        noisy_sim   = AerSimulator(noise_model=nm)
        qc_t_noisy  = transpile(circuit_m, basis_gates=['cx','rz','rx','h','x'],
                                 optimization_level=3, seed_transpiler=cfg.seed_transpiler)
        noisy_counts = noisy_sim.run(qc_t_noisy, shots=cfg.shots,
                                      seed_simulator=cfg.seed).result().get_counts()
        noisy_avg   = sum((cnt/cfg.shots)*cfg.cost_fn(bs) for bs,cnt in noisy_counts.items())
        G           = qc_t_noisy.count_ops().get('cx',0)
        C_bar       = C_opt/2.0
        noise_pred  = (1-cfg.noise_eps)**G*best_F + (1-(1-cfg.noise_eps)**G)*C_bar

    # Step 9: Hardware transpilation
    transpiled_qc, cx_reduction = None, None
    if cfg.backend is not None:
        pm            = generate_preset_pass_manager(backend=cfg.backend, optimization_level=3,
                                                     layout_method='sabre', routing_method='sabre',
                                                     seed_transpiler=cfg.seed_transpiler)
        transpiled_qc = pm.run(circuit_m)
        logical_cx    = circuit_stats(circuit_m)['two_qubit_gates']
        opt_cx        = circuit_stats(transpiled_qc)['two_qubit_gates']
        cx_reduction  = (logical_cx-opt_cx)/logical_cx if logical_cx>0 else 0.0

    # Step 10: INTERP
    interp_next = interp_init_general(best_params)
    elapsed     = time.perf_counter()-t0

    return QAOAResult(
        optimal_params=best_params, optimal_F=best_F, alpha=alpha,
        gradient_at_opt=grad, statevector=sv, prob_distribution=prob_dist,
        top_k_bitstrings=ranked[:10], shot_counts_ideal=ideal_counts,
        shot_avg_cost_ideal=ideal_avg, shot_counts_noisy=noisy_counts,
        shot_avg_cost_noisy=noisy_avg, noise_prediction=noise_pred,
        transpiled_circuit=transpiled_qc, transpilation_cx_reduction=cx_reduction,
        interp_params_next=interp_next, elapsed_time=elapsed,
        n_circuit_evaluations=n_evals, brute_force_optimum=bf_opt,
        brute_force_solutions=bf_sols)

print('run_qaoa_general() pipeline defined.')


### 15.6 Monotonicity Sweep — Survey Theorem 3.3

In [ ]:
def run_qaoa_sweep_general(cfg: QAOAConfig, p_range):
    """INTERP-chained sweep over depths; verifies M_p >= M_{p-1}."""
    results, warm = [], None
    for p in p_range:
        cfg_p = QAOAConfig(
            H_C=cfg.H_C, cost_fn=cfg.cost_fn, n_qubits=cfg.n_qubits, p=p,
            mixer=cfg.mixer, optimizer=cfg.optimizer, n_restarts=cfg.n_restarts,
            warm_start_params=warm, C_opt=cfg.C_opt, backend=cfg.backend,
            noise_eps=cfg.noise_eps, shots=cfg.shots, seed=cfg.seed,
            seed_transpiler=cfg.seed_transpiler, param_bounds=cfg.param_bounds)
        result = run_qaoa_general(cfg_p)
        results.append(result)
        warm = result.interp_params_next
        if len(results)>=2:
            assert results[-1].optimal_F >= results[-2].optimal_F-1e-6, \
                f'Monotonicity violated at p={p}'
    return results

print('run_qaoa_sweep_general() defined.')


### 15.7 Problem Encoder Library

In [ ]:
# ── Max-Cut (weighted / unweighted) ─────────────────────────────────
def maxcut_hamiltonian_general(edges, n_qubits, weights=None):
    if weights is None: weights = [1.0]*len(edges)
    terms = [('I'*n_qubits, 0.5*sum(weights))]
    for (i,j),w in zip(edges,weights):
        label = ['I']*n_qubits
        label[n_qubits-1-i]='Z'; label[n_qubits-1-j]='Z'
        terms.append((''.join(label), -0.5*w))
    return SparsePauliOp.from_list(terms)

def maxcut_cost_fn_general(edges, weights=None):
    if weights is None: weights = [1.0]*len(edges)
    def cost(bs):
        bits = bs[::-1]
        return sum(w for (i,j),w in zip(edges,weights) if bits[i]!=bits[j])
    return cost

# ── Number Partitioning ───────────────────────────────────────────────
def number_partition_hamiltonian(values):
    n     = len(values)
    terms = []
    for i,vi in enumerate(values):
        for j,vj in enumerate(values):
            if i==j:
                terms.append(('I'*n, -vi*vj))
            else:
                label=['I']*n; label[n-1-i]='Z'; label[n-1-j]='Z'
                terms.append((''.join(label), -vi*vj))
    return SparsePauliOp.from_list(terms).simplify()

def number_partition_cost_fn(values):
    def cost(bs):
        bits = bs[::-1]
        S    = sum(v for v,b in zip(values,bits) if b=='0')
        Sbar = sum(v for v,b in zip(values,bits) if b=='1')
        return -(S-Sbar)**2
    return cost

print('Problem encoder library defined: Max-Cut, Number Partitioning.')


### 15.8 Example 1: C4 Max-Cut via Generalised Pipeline
Reproduces the paper's results exactly using the general framework.

In [ ]:
print('Example 1: C4 Max-Cut (reproducing paper results via general framework)')
print('='*60)

H_C_gen   = maxcut_hamiltonian_general(EDGES, NUM_QUBITS)
cost_gen  = maxcut_cost_fn_general(EDGES)

cfg_c4 = QAOAConfig(
    H_C=H_C_gen, cost_fn=cost_gen, n_qubits=NUM_QUBITS,
    p=1, mixer='X', optimizer='COBYLA', n_restarts=5,
    noise_eps=2e-3, shots=4096, seed=42)

result_c4 = run_qaoa_general(cfg_c4)
p = len(result_c4.optimal_params)//2
print(f'  Optimal F     : {result_c4.optimal_F:.6f}  (expected 3.000000)')
print(f'  Alpha         : {result_c4.alpha:.6f}  (expected 0.750000)')
print(f'  |grad| at opt : {np.linalg.norm(result_c4.gradient_at_opt):.2e}  (expected ≈ 0)')
print(f'  BF optimum    : {result_c4.brute_force_optimum}')
print(f'  BF solutions  : {result_c4.brute_force_solutions}')
print(f'  Ideal ⟨C⟩     : {result_c4.shot_avg_cost_ideal:.4f}')
if result_c4.shot_avg_cost_noisy:
    print(f'  Noisy ⟨C⟩     : {result_c4.shot_avg_cost_noisy:.4f}')
    print(f'  Noise pred.   : {result_c4.noise_prediction:.4f}')
print(f'  INTERP init   : {result_c4.interp_params_next}')
print(f'  Elapsed time  : {result_c4.elapsed_time:.2f} s')
print()
print('Top-5 bitstrings:')
for bs,cost,prob in result_c4.top_k_bitstrings[:5]:
    print(f'  {bs}  cost={cost:.1f}  prob={prob:.4f}')


### 15.9 Example 1b: Monotonicity Sweep p=1,2,3

In [ ]:
sweep_cfg = QAOAConfig(
    H_C=H_C_gen, cost_fn=cost_gen, n_qubits=NUM_QUBITS,
    optimizer='COBYLA', n_restarts=5, shots=4096, seed=42)

sweep = run_qaoa_sweep_general(sweep_cfg, p_range=[1,2,3])
print('Monotonicity sweep:')
for i,r in enumerate(sweep,1):
    print(f'  p={i}: F={r.optimal_F:.6f}  α={r.alpha:.6f}  t={r.elapsed_time:.2f}s')
Fs = [r.optimal_F for r in sweep]
print(f'\nMonotonicity {Fs[0]:.3f} ≤ {Fs[1]:.3f} ≤ {Fs[2]:.3f}: ',
      '✓' if Fs[0]<=Fs[1]<=Fs[2]+1e-6 else '✗')


### 15.10 Example 2: Weighted Max-Cut on K4

In [ ]:
print('Example 2: Weighted Max-Cut on K4 (complete 4-graph)')
print('='*60)

EDGES_K4   = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
WEIGHTS_K4 = [2.0, 1.0, 3.0, 1.5, 0.5, 2.5]

H_C_wmc  = maxcut_hamiltonian_general(EDGES_K4, 4, WEIGHTS_K4)
cost_wmc = maxcut_cost_fn_general(EDGES_K4, WEIGHTS_K4)

cfg_wmc = QAOAConfig(
    H_C=H_C_wmc, cost_fn=cost_wmc, n_qubits=4,
    p=1, optimizer='COBYLA', n_restarts=5, shots=4096, seed=42)

result_wmc = run_qaoa_general(cfg_wmc)
print(f'  BF optimum : {result_wmc.brute_force_optimum}  solutions: {result_wmc.brute_force_solutions}')
print(f'  F_p*       : {result_wmc.optimal_F:.6f}')
print(f'  Alpha      : {result_wmc.alpha:.6f}')
print(f'  |grad|     : {np.linalg.norm(result_wmc.gradient_at_opt):.2e}')


### 15.11 Example 3: Number Partitioning

In [ ]:
print('Example 3: Number Partitioning  values=[3,1,1,2,2,1]')
print('='*60)

VALUES  = [3.0, 1.0, 1.0, 2.0, 2.0, 1.0]
N_NP    = len(VALUES)

H_C_np  = number_partition_hamiltonian(VALUES)
cost_np = number_partition_cost_fn(VALUES)

cfg_np = QAOAConfig(
    H_C=H_C_np, cost_fn=cost_np, n_qubits=N_NP,
    p=2, optimizer='COBYLA', n_restarts=5, shots=4096, seed=42)

result_np = run_qaoa_general(cfg_np)
print(f'  BF optimum : {result_np.brute_force_optimum}  (0 = perfect partition)')
print(f'  F_p*       : {result_np.optimal_F:.6f}')
print(f'  Alpha      : {result_np.alpha:.6f}')
print()
print('  Top-5 partitions:')
for bs,c,prob in result_np.top_k_bitstrings[:5]:
    grp0 = [v for v,b in zip(VALUES,bs[::-1]) if b=='0']
    grp1 = [v for v,b in zip(VALUES,bs[::-1]) if b=='1']
    print(f'    {bs}  |S|={sum(grp0):.0f}, |S̄|={sum(grp1):.0f}  cost={c:.1f}  prob={prob:.4f}')
